In [220]:
import pandas as pd
from rapidfuzz import fuzz, process, utils
from pathlib import Path

In [ ]:
translated_gsa_path = '/Users/ayshahchan/Desktop/PhD/eurocropsV2/EuroCropsV2/data/eurocrop_trans/at_trans_stats.csv'
mapped_output_path = '/Users/ayshahchan/Desktop/PhD/eurocropsV2/EuroCropsV2/data/eurocrop_trans/mapping/at_mapped.csv'


In [210]:
translated_gsa = pd.read_csv(translated_gsa_path)
hcat4_ref = pd.read_csv('/Users/ayshahchan/Desktop/PhD/eurocropsV2/EuroCropsV2/data/cropcodemapping/hcat4.csv')


In [211]:
name_to_code = dict(zip(hcat4_ref['hcat4_name'], hcat4_ref['hcat4_code']))


In [212]:
trans_name = translated_gsa['translated_name']
# if all in one file, can also filter by nuts code
# nuts_code = 'de4'
# trans_name = translated_gsa[translated_gsa['nuts']==nuts_code]['translated_name']

In [213]:
candidates = hcat4_ref['hcat4_name'].str.replace("_", " ", regex=False)

def find_best_match(query, candidates):
    match = process.extractOne(query, candidates, scorer=fuzz.WRatio, processor=utils.default_process)
    # fuzz.WRatio is a weighted combination of multiple Levenshtein-based similarity 
    if match is None:
        return None
    else:
    # score = match[1]
        best_match = match[0]
        return best_match

In [214]:
# Apply to series
series_matched = trans_name.apply(lambda x: find_best_match(x, candidates))
series_matched =  series_matched.str.replace(" ", "_", regex=False)

In [215]:
translated_gsa['hcat4_name_auto'] = series_matched
translated_gsa['hcat4_code_auto'] = translated_gsa['hcat4_name_auto'].map(name_to_code)

In [216]:
translated_gsa.to_csv(mapped_output_path, index=False)







In [222]:
files = Path("/Users/ayshahchan/Desktop/PhD/eurocropsV2/EuroCropsV2/data/eurocrop_trans/mapping/").glob("*_mapped.csv")
# ref_mapped_gsa_path ='/Users/ayshahchan/Desktop/PhD/eurocropsV2/EuroCropsV2/data/eurocrop_trans/mapping/at_mapped.csv'
for ref_mapped_gsa_path in files:    
    mapped_gsa = pd.read_csv(ref_mapped_gsa_path)
    country_code = mapped_gsa['nuts'].iloc[0].upper() 
    hcat4_name = mapped_gsa['hcat4_name']
    series_matched = mapped_gsa['hcat4_name_auto']
    # hcat4_name = mapped_gsa[mapped_gsa['nuts']==nuts_code]['hcat4_name']
    accuracy = sum(series_matched == hcat4_name)/len(hcat4_name) *100
    print(f"{country_code}'s automatic matching accuracy: {accuracy:.2f}%")

PT's automatic matching accuracy: 41.53%
IE's automatic matching accuracy: 51.63%
ES's automatic matching accuracy: 53.55%
BE3's automatic matching accuracy: 60.48%
ITI1's automatic matching accuracy: 53.93%
EE's automatic matching accuracy: 54.14%
NL's automatic matching accuracy: 51.32%
DK's automatic matching accuracy: 42.24%
BE2's automatic matching accuracy: 49.59%
BG's automatic matching accuracy: 61.17%
DE4's automatic matching accuracy: 56.20%
SI's automatic matching accuracy: 62.09%
FR's automatic matching accuracy: 51.31%
SK's automatic matching accuracy: 69.26%
CZ's automatic matching accuracy: 59.09%
AT's automatic matching accuracy: 50.61%
FI's automatic matching accuracy: 52.46%
